 # Regression pipeline for predicting Oral Temperature from Infrared Thermography data
 ## (Leakage-corrected version)

 Changes vs. the original notebook, and why:

 1. **Subject-level leakage** — Rounds are repeated measurements of the *same* people
    ~10 minutes apart. Splitting "Rounds 1-3 = train, Round 4 = test" lets the model see
    every test subject during training. Fixed by combining all four rounds into one table
    and doing a **GroupShuffleSplit on `Subject_ID`**, so no person appears in both sets.
 2. **Model selection leakage** — the original code scored all 10 models on the test set
    and then picked the "best" one from that same test comparison. Fixed by selecting the
    winning model via cross-validation on the training set only; the test set is touched
    exactly once, for the final reported numbers.
 3. **Fever-range filtering removed from the test set** — filtering out `True_Core_Temp_Fast
    > 39.5` (using the wrong target column) stripped febrile cases from evaluation, which is
    exactly the range a fever-screening model needs to be judged on. Fixed by only filtering
    the *training* data (using the actual modeling target) and reporting test performance
    on the full, unfiltered test set, plus separately on any fever cases within it.
 4. **Outlier bounds computed per-round including the test round** — IQR bounds are now
    computed on the training split only and applied to both train and test.
 5. **Ablation study used mismatched hyperparameters** for the "with" vs. "without"
    Thermal_Gradient comparison. Fixed by using identical hyperparameters in both arms.

 Everything else (renaming, EDA, correlation filtering fit on train only, scaler/encoder
 fit on train only) was already implemented correctly and is preserved.

 0. Import all necessary modules

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    GroupShuffleSplit,
    GroupKFold,
    cross_val_score
)

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

RANDOM_STATE = 42

 1. Load the dataset. Each of the 4 sheets is one trial/round for the same cohort of
 subjects, ~10 minutes apart.

In [ ]:
df = [
    pd.read_excel('../datasets/ICI.xlsx', sheet_name='Round 1', header=2),
    pd.read_excel('../datasets/ICI.xlsx', sheet_name='Round 2', header=2),
    pd.read_excel('../datasets/ICI.xlsx', sheet_name='Round 3', header=2),
    pd.read_excel('../datasets/ICI.xlsx', sheet_name='Round 4', header=2),
]
df[0].head(10)

 2. Rename columns to legible names (unchanged from original).

In [ ]:
base_mapping = {
    'T_offset': 'Calibration_Offset',
    'Max1R13_': 'Right_Inner_Eye_Max',
    'Max1L13_': 'Left_Inner_Eye_Max',
    'aveAllR13_': 'Right_Inner_Eye_Avg',
    'aveAllL13_': 'Left_Inner_Eye_Avg',
    'T_RC': 'Right_Eye_Avg',
    'T_RC_Dry': 'Right_Eye_Skin',
    'T_RC_Wet': 'Right_Tear_Duct',
    'T_RC_Max': 'Right_Eye_Max',
    'T_LC': 'Left_Eye_Avg',
    'T_LC_Dry': 'Left_Eye_Skin',
    'T_LC_Wet': 'Left_Tear_Duct',
    'T_LC_Max': 'Left_Eye_Max',
    'RCC': 'Right_Eye_Center',
    'LCC': 'Left_Eye_Center',
    'canthiMax': 'Hottest_Eye_Overall',
    'canthi4Max': 'Hottest_Eye_4Points',
    'T_FHCC': 'Forehead_Center',
    'T_FHRC': 'Forehead_Right',
    'T_FHLC': 'Forehead_Left',
    'T_FHBC': 'Forehead_Bottom',
    'T_FHTC': 'Forehead_Top',
    'T_FH_Max': 'Forehead_Overall_Max',
    'T_FHC_Max': 'Forehead_Center_Max',
    'T_Max': 'Face_Overall_Max',
    'T_OR': 'Mouth_Avg',
    'T_OR_Max': 'Mouth_Max'
}
full_mapping = {
    'SubjectID': 'Subject_ID',
    'aveOralF': 'True_Core_Temp_Fast',
    'aveOralM': 'True_Core_Temp_Slow',
    'T_atm': 'Room_Temperature',
    'Humidity': 'Room_Humidity',
    'Distance': 'Camera_Distance_m',
    'Cosmetics': 'Wearing_Makeup'
}
for old_base, new_base in base_mapping.items():
    for round_num in range(1, 5):
        old_col_name = f"{old_base}{round_num}"
        new_col_name = f"{new_base}_R{round_num}"
        full_mapping[old_col_name] = new_col_name

for i in range(len(df)):
    df[i].rename(columns=full_mapping, inplace=True)
df[0].head()

 3-4. Column listing / dtype identification (unchanged).

In [ ]:
print(df[1].columns.tolist())
df[0].info()
df[0].dtypes

In [ ]:
nominal_cols = ['Subject_ID', 'Gender', 'Ethnicity', 'Wearing_Makeup']
ordinal_cols = ['Age']
datetime_cols = ['Time', 'Date']
all_cols = df[3].columns.tolist()
exclude_cols = nominal_cols + ordinal_cols + datetime_cols
numerical_cols_all = [col for col in all_cols if col not in exclude_cols]
print(f" NOMINAL COLUMNS ({len(nominal_cols)})")
print(nominal_cols)
print(f"\nORDINAL COLUMNS ({len(ordinal_cols)})")
print(ordinal_cols)
print(f"\n NUMERICAL COLUMNS ({len(numerical_cols_all)})")
print(numerical_cols_all)

 ### Drop rows with missing values (~0.5% of data, unchanged rationale from original).

In [ ]:
df[0].isna().sum()

In [ ]:
for i in range(len(df)):
    print(f"Round {i+1} duplicates:", df[i].duplicated().sum())

In [ ]:
for i in range(len(df)):
    df[i].dropna(inplace=True)

 Count of Male vs Female (unchanged EDA).

In [ ]:
gender_ethnicity_counts = (
    df[0][df[0]["Gender"].notnull() & df[0]["Ethnicity"].notnull()]
    [["Subject_ID", "Gender", "Ethnicity"]]
    .groupby(["Gender", "Ethnicity"], as_index=False)
    .count()
    .rename(columns={"Subject_ID": "Count"})
)

plt.figure()
sns.barplot(data=gender_ethnicity_counts, x="Gender", y="Count", hue="Ethnicity", palette="Set2")
plt.title("Participant Count by Gender and Ethnicity")
plt.xlabel("Gender")
plt.ylabel("Count")
plt.legend(title="Ethnicity")
plt.show()

print(df[0]["Gender"].value_counts())

 ## FIX #1 — Check subject overlap across rounds before deciding how to split

 This is the step the original notebook skipped. If the same `Subject_ID` appears in
 more than one round, a round-based train/test split leaks that subject's identity
 (and hence their personal baseline temperature) across the split.

In [ ]:
subject_sets = [set(df[i]['Subject_ID']) for i in range(4)]
for i in range(4):
    for j in range(i + 1, 4):
        overlap = subject_sets[i] & subject_sets[j]
        print(f"Round {i+1} vs Round {j+1}: {len(overlap)} shared subjects "
              f"out of {len(subject_sets[i])}/{len(subject_sets[j])}")

all_subjects_union = set.union(*subject_sets)
all_subjects_intersection = set.intersection(*subject_sets)
print(f"\nTotal unique subjects across all rounds: {len(all_subjects_union)}")
print(f"Subjects present in ALL 4 rounds: {len(all_subjects_intersection)}")

 In this dataset the rounds are repeated measurements of the same cohort, so we expect
 heavy overlap. **Regardless of the overlap result above, we use a subject-grouped split**
 rather than a round-based split — this is the only split that is guaranteed safe whether
 overlap is partial or total, and it costs nothing if overlap happens to be zero.

 ### Boxplots / histograms / KDE / heatmap EDA — unchanged from original, kept for context

In [ ]:
data = df[0].copy()
melted_data = data[['True_Core_Temp_Fast', 'True_Core_Temp_Slow']].melt(
    var_name='Sensor Type', value_name='Temperature'
)
plt.figure()
sns.kdeplot(data=melted_data, x='Temperature', hue='Sensor Type', fill=True, palette='crest')
plt.axvline(x=38.0, color='red', linestyle='--', linewidth=2, label='Fever Threshold')
plt.title('Core Temperature Distribution (Round 1)')
plt.xlabel('Temperature')
plt.legend()
plt.show()

 (Boxplot-by-region, histogram, and pairplot EDA cells from the original notebook are
 unchanged and omitted here only for brevity — they involve no leakage since they're
 descriptive-only and don't feed into modeling decisions.)

 ## Build one combined table across all four rounds (instead of Round1-3 vs Round4)

 Each round already shares the same schema once the `_R{n}` suffix is stripped. Instead
 of treating Round 4 as "the test set", we stack all four rounds long-wise and let the
 subject-grouped split decide which *subjects'* rows go to train vs. test. A `Round`
 column is kept so we can still inspect round effects if needed, but it is not used as
 the split criterion.

In [ ]:
rounds_stripped = []
for i in range(4):
    r = df[i].copy()
    r.columns = r.columns.str.replace(f'_R{i+1}', '', regex=False)
    r['Round'] = i + 1
    rounds_stripped.append(r)

combined_data = pd.concat(rounds_stripped, ignore_index=True)
print("Combined shape:", combined_data.shape)
print("Unique subjects in combined data:", combined_data['Subject_ID'].nunique())

 ## FIX #1 (continued) — Subject-grouped train/test split

 80/20 split by `Subject_ID`: every row belonging to a given subject goes entirely to
 either train or test, never both.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(
    combined_data, groups=combined_data['Subject_ID']
))

train_data = combined_data.iloc[train_idx].copy().reset_index(drop=True)
test_data = combined_data.iloc[test_idx].copy().reset_index(drop=True)

print("Training shape:", train_data.shape, "| unique subjects:", train_data['Subject_ID'].nunique())
print("Testing shape:", test_data.shape, "| unique subjects:", test_data['Subject_ID'].nunique())

overlap_check = set(train_data['Subject_ID']) & set(test_data['Subject_ID'])
assert len(overlap_check) == 0, "Subject leakage between train and test!"
print("Confirmed: zero subject overlap between train and test.")

 ## FIX #4 — Outlier filtering: bounds computed on TRAIN only, applied to both

 The original notebook computed IQR bounds separately per round, including Round 4
 (the test set), which lets the test set's own distribution shape its own cleaning.
 Here bounds come from `train_data` only and are then applied identically to `test_data`.

In [ ]:
env_cols = ['Room_Temperature', 'Room_Humidity', 'Camera_Distance_m']

env_bounds = {}
for col in env_cols:
    Q1 = train_data[col].quantile(0.25)
    Q3 = train_data[col].quantile(0.75)
    IQR = Q3 - Q1
    env_bounds[col] = (Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

print("Bounds derived from TRAIN data only:")
for col, (lo, hi) in env_bounds.items():
    print(f"  {col}: [{lo:.2f}, {hi:.2f}]")

for label, dataset in [("train", train_data), ("test", test_data)]:
    before = len(dataset)
    mask = pd.Series(True, index=dataset.index)
    for col, (lo, hi) in env_bounds.items():
        mask &= dataset[col].between(lo, hi)
    dataset_filtered = dataset[mask]
    print(f"{label}: {before} -> {len(dataset_filtered)} rows after applying train-derived bounds")
    if label == "train":
        train_data = dataset_filtered.reset_index(drop=True)
    else:
        test_data_env_filtered = dataset_filtered.reset_index(drop=True)

# Keep an UNFILTERED copy of test_data for the final, honest evaluation (FIX #3),
# and a filtered copy for anyone who wants an apples-to-apples comparison with train
# preprocessing.
test_data_unfiltered = test_data.copy()
test_data = test_data_env_filtered

 ## FIX #3 — Fever-range filtering: training-only, consistent target column

 The original notebook filtered `True_Core_Temp_Fast <= 39.5` from *every* round,
 including the test round, and used the Fast column even though the model targets
 the Slow column. Fever-range cases are exactly what a screening model should be
 evaluated on, so:
   - We filter using the actual modeling target (`True_Core_Temp_Slow`).
   - We only apply this filter to the TRAINING data (to stabilize model fitting against
     a handful of extreme high-temperature points).
   - The TEST set is left with its full temperature range, including fevers, and we
     report metrics on the full test set plus a fever-only subset for transparency.

In [ ]:
TARGET_COL = 'True_Core_Temp_Slow'
FEVER_TRAIN_CUTOFF = 39.5

print("Train records before fever cutoff:", len(train_data))
train_data = train_data[train_data[TARGET_COL] <= FEVER_TRAIN_CUTOFF].reset_index(drop=True)
print("Train records after fever cutoff:", len(train_data))

fever_threshold_eval = 38.0
n_fever_test = (test_data[TARGET_COL] >= fever_threshold_eval).sum()
print(f"Test set retains {len(test_data)} rows, including {n_fever_test} "
      f"at/above the {fever_threshold_eval}C fever threshold.")

 ## FIX #5 (setup) — Correlation-based feature dropping, fit on TRAIN only (unchanged
 logic from the original, which was already correct — kept here for completeness).

In [ ]:
numerical_features = [
    'Calibration_Offset', 'Right_Inner_Eye_Max', 'Left_Inner_Eye_Max',
    'Right_Inner_Eye_Avg', 'Left_Inner_Eye_Avg', 'Right_Eye_Avg',
    'Right_Eye_Skin', 'Right_Tear_Duct', 'Right_Eye_Max', 'Left_Eye_Avg',
    'Left_Eye_Skin', 'Left_Tear_Duct', 'Left_Eye_Max', 'Right_Eye_Center',
    'Left_Eye_Center', 'Hottest_Eye_Overall', 'Hottest_Eye_4Points',
    'Forehead_Center', 'Forehead_Right', 'Forehead_Left', 'Forehead_Bottom',
    'Forehead_Top', 'Forehead_Overall_Max', 'Forehead_Center_Max',
    'Face_Overall_Max', 'Mouth_Avg', 'Mouth_Max',
    'Room_Temperature', 'Room_Humidity', 'Camera_Distance_m'
]

threshold = 0.9
corr_matrix = train_data[numerical_features].corr().abs()
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper_triangle.columns if any(upper_triangle[col] > threshold)]

print(f"Features to drop ({len(to_drop)}): {to_drop}")

train_data = train_data.drop(columns=to_drop)
test_data = test_data.drop(columns=to_drop)
test_data_unfiltered = test_data_unfiltered.drop(columns=to_drop)

plt.figure(figsize=(14, 12))
sns.heatmap(corr_matrix, cmap='coolwarm', annot=False, square=True,
            cbar_kws={'label': 'Absolute Correlation'})
plt.title("Correlation Heatmap - Numerical Features (train only, before redundancy filtering)")
plt.tight_layout()
plt.show()

 ### Feature engineering — Thermal_Gradient (unchanged rationale)

In [ ]:
for dataset in (train_data, test_data, test_data_unfiltered):
    dataset['Thermal_Gradient'] = dataset['Forehead_Overall_Max'] - dataset['Mouth_Avg']

 ## Note on `Calibration_Offset`

 Before trusting this as a predictor, confirm it is a sensor/ambient calibration term
 and not derived from the oral-temperature target. If it were computed from
 `True_Core_Temp_Fast`/`True_Core_Temp_Slow`, it would leak the target directly.

In [ ]:
calib_corr = train_data[['Calibration_Offset', 'True_Core_Temp_Fast', 'True_Core_Temp_Slow']].corr()
print(calib_corr)
print("\nIf Calibration_Offset correlates near +/-1.0 with either target column above,")
print("treat it as a leak and drop it — do not proceed with it as a feature.")

 ## Assemble modeling tables

In [ ]:
feature_cols = [
    'Calibration_Offset', 'Right_Inner_Eye_Max', 'Left_Inner_Eye_Max',
    'Right_Inner_Eye_Avg', 'Left_Inner_Eye_Avg', 'Forehead_Center',
    'Forehead_Right', 'Forehead_Left', 'Forehead_Top', 'Forehead_Overall_Max',
    'Mouth_Avg', 'Room_Temperature', 'Room_Humidity', 'Camera_Distance_m',
    'Thermal_Gradient', 'Gender', 'Ethnicity', 'Wearing_Makeup'
]
target_col = TARGET_COL

train_model_data = train_data[feature_cols + [target_col]].dropna()
test_model_data = test_data[feature_cols + [target_col]].dropna()
test_model_data_unfiltered = test_data_unfiltered[feature_cols + [target_col]].dropna()

X_train = train_model_data[feature_cols]
y_train = train_model_data[target_col]

X_test = test_model_data[feature_cols]
y_test = test_model_data[target_col]

X_test_full = test_model_data_unfiltered[feature_cols]
y_test_full = test_model_data_unfiltered[target_col]

print(f"Training samples: {len(X_train)}")
print(f"Testing samples (env-outlier-filtered): {len(X_test)}")
print(f"Testing samples (fully unfiltered, incl. fevers): {len(X_test_full)}")

numerical_cols = [c for c in feature_cols if c not in ['Gender', 'Ethnicity', 'Wearing_Makeup']]
categorical_cols = ['Gender', 'Ethnicity', 'Wearing_Makeup']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
X_test_full_processed = preprocessor.transform(X_test_full)

encoded_feature_names = preprocessor.get_feature_names_out()
print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

 ## FIX #2 — Model selection via cross-validation on TRAIN ONLY

 The original notebook fit every model, scored all 10 directly against `y_test`, ranked
 them by test R², and then declared a "best model" from that ranking — using the test
 set to *choose* the model, not just evaluate it. Below, hyperparameter tuning and model
 comparison both happen via K-fold CV on the training data. The test set is used exactly
 once at the end, for the final reported metrics of the CV-selected winner.

In [ ]:
group_kf = GroupKFold(n_splits=5)
cv_groups = train_model_data.loc[X_train.index, :].index  # placeholder if Subject_ID retained
# If Subject_ID was kept through to train_model_data, use grouped CV; otherwise fall back
# to a plain KFold. (Subject_ID is intentionally excluded from feature_cols above, so pull
# it back in for CV grouping only.)
train_subject_ids = train_data.loc[train_model_data.index, 'Subject_ID']
kf = GroupKFold(n_splits=5)

candidate_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(max_iter=10000),
    "ElasticNet Regression": ElasticNet(max_iter=10000, random_state=RANDOM_STATE),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random Forest Regressor": RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting Regressor": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "Support Vector Regressor (SVR)": SVR(),
    "K-Nearest Neighbors Regressor": KNeighborsRegressor(),
}

param_grids = {
    "Ridge Regression": {'alpha': [0.01, 0.1, 1.0, 10.0, 50.0, 100.0]},
    "Lasso Regression": {'alpha': [0.001, 0.01, 0.1, 1.0]},
    "ElasticNet Regression": {'alpha': [0.001, 0.01, 0.1, 1.0], 'l1_ratio': [0.2, 0.5, 0.8]},
    "Decision Tree Regressor": {'max_depth': [5, 10, 15, None]},
    "Random Forest Regressor": {'n_estimators': [100, 300, 500], 'max_depth': [5, 10, 15, None]},
    "Gradient Boosting Regressor": {'n_estimators': [100, 300], 'learning_rate': [0.03, 0.05, 0.1], 'max_depth': [2, 3, 4]},
    "Support Vector Regressor (SVR)": {'C': [1, 10, 50], 'gamma': ['scale', 'auto']},
    "K-Nearest Neighbors Regressor": {'n_neighbors': [3, 5, 10, 15]},
}

fitted_models = {}
cv_results = []

for name, base_model in candidate_models.items():
    if name in param_grids:
        grid = GridSearchCV(
            base_model, param_grids[name], cv=kf, scoring='r2', n_jobs=-1
        )
        grid.fit(X_train_processed, y_train, groups=train_subject_ids)
        best_model = grid.best_estimator_
        best_cv_r2 = grid.best_score_
        best_params = grid.best_params_
    else:
        scores = cross_val_score(
            base_model, X_train_processed, y_train, cv=kf,
            groups=train_subject_ids, scoring='r2', n_jobs=-1
        )
        best_model = base_model.fit(X_train_processed, y_train)
        best_cv_r2 = scores.mean()
        best_params = {}

    fitted_models[name] = best_model
    cv_results.append({"Model": name, "CV_R2_mean": best_cv_r2, "Best_Params": best_params})
    print(f"{name}: CV R2 = {best_cv_r2:.4f}  params={best_params}")

# Polynomial regression handled separately (degree chosen via CV, not via test R2)
poly_cv_scores = {}
for degree in [2, 3]:
    poly_pipeline = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('linear', LinearRegression())
    ])
    scores = cross_val_score(
        poly_pipeline, X_train_processed, y_train, cv=kf,
        groups=train_subject_ids, scoring='r2', n_jobs=-1
    )
    poly_cv_scores[degree] = scores.mean()
    print(f"Polynomial degree {degree}: CV R2 = {scores.mean():.4f}")

best_degree = max(poly_cv_scores, key=poly_cv_scores.get)
polynomial_model = Pipeline([
    ('poly', PolynomialFeatures(degree=best_degree, include_bias=False)),
    ('linear', LinearRegression())
]).fit(X_train_processed, y_train)
fitted_models["Polynomial Regression"] = polynomial_model
cv_results.append({"Model": "Polynomial Regression", "CV_R2_mean": poly_cv_scores[best_degree],
                    "Best_Params": {"degree": best_degree}})

cv_results_df = pd.DataFrame(cv_results).sort_values("CV_R2_mean", ascending=False).reset_index(drop=True)
cv_results_df

 The model with the best **cross-validated training** R² is selected as the winner.
 This is the only point in the pipeline that determines "best model" — no test-set
 information is used.

In [ ]:
best_model_name = cv_results_df.loc[0, "Model"]
best_model = fitted_models[best_model_name]
print("Selected model (via CV on train only):", best_model_name)

 ## Final, single evaluation on the held-out test set

 Reported twice: on the env-outlier-filtered test set (comparable cleaning to train) and
 on the fully unfiltered test set including fever-range cases, since that's the range a
 real screening use case needs to be judged on.

In [ ]:
def evaluate(model, X, y, label):
    preds = model.predict(X)
    r2 = r2_score(y, preds)
    rmse = np.sqrt(mean_squared_error(y, preds))
    mae = mean_absolute_error(y, preds)
    print(f"[{label}] R2={r2:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}  n={len(y)}")
    return preds, {"R2": r2, "RMSE": rmse, "MAE": mae}

test_preds, test_metrics = evaluate(best_model, X_test_processed, y_test, "Test (filtered)")
test_full_preds, test_full_metrics = evaluate(best_model, X_test_full_processed, y_test_full, "Test (full, incl. fevers)")

fever_mask = y_test_full >= fever_threshold_eval
if fever_mask.sum() > 0:
    evaluate(best_model, X_test_full_processed[fever_mask.values], y_test_full[fever_mask], "Test (fever cases only)")
else:
    print("No fever-range cases present in the held-out test subjects.")

 ### Predicted vs actual, residuals (on the full, unfiltered test set — the honest number)

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test_full, y=test_full_preds, alpha=0.6)
lims = [min(y_test_full.min(), test_full_preds.min()), max(y_test_full.max(), test_full_preds.max())]
plt.plot(lims, lims, linestyle='--', linewidth=2, label='Perfect Prediction')
plt.xlabel("Actual True Core Temperature")
plt.ylabel("Predicted True Core Temperature")
plt.title(f"Predicted vs Actual (full test set) - {best_model_name}")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
residuals_full = y_test_full - test_full_preds
plt.figure(figsize=(8, 6))
sns.scatterplot(x=test_full_preds, y=residuals_full, alpha=0.6)
plt.axhline(y=0, linestyle='--', linewidth=2, label='Zero Residual')
plt.xlabel("Predicted True Core Temperature")
plt.ylabel("Residual")
plt.title(f"Residual Plot (full test set) - {best_model_name}")
plt.legend()
plt.tight_layout()
plt.show()

 ### Random Forest feature importance (only meaningful if RF was the selected/fitted model;
 shown here regardless for comparison, using the CV-tuned RF from `fitted_models`).

In [ ]:
rf_for_importance = fitted_models["Random Forest Regressor"]
feature_importance = pd.DataFrame({
    "Feature": encoded_feature_names,
    "Importance": rf_for_importance.feature_importances_
}).sort_values("Importance", ascending=False)

plt.figure(figsize=(10, 7))
sns.barplot(data=feature_importance, x="Importance", y="Feature")
plt.title("Random Forest Feature Importance (CV-tuned, fit on train only)")
plt.tight_layout()
plt.show()

 ## FIX #5 — Ablation: effect of Thermal_Gradient, same hyperparameters in both arms

 The original notebook compared *tuned* models (with the feature) against *default-
 hyperparameter* models (without the feature), so the "improvement" partly reflected
 tuning effort, not the feature itself. Here both arms reuse the exact same
 hyperparameters — taken from the CV step above — for every model.

In [ ]:
feature_cols_without = [c for c in feature_cols if c != 'Thermal_Gradient']
numerical_cols_without = [c for c in numerical_cols if c != 'Thermal_Gradient']

X_train_without_raw = train_model_data[feature_cols_without]
X_test_without_raw = test_model_data_unfiltered[feature_cols_without]  # honest test set
y_test_without = test_model_data_unfiltered[target_col]

preprocessor_without = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols_without),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
])
X_train_without = preprocessor_without.fit_transform(X_train_without_raw)
X_test_without = preprocessor_without.transform(X_test_without_raw)

def clone_with_best_params(name):
    """Recreate an estimator with the exact hyperparameters selected during CV."""
    params = cv_results_df.loc[cv_results_df["Model"] == name, "Best_Params"].iloc[0]
    base = {
        "Linear Regression": LinearRegression(),
        "Ridge Regression": Ridge(),
        "Lasso Regression": Lasso(max_iter=10000),
        "ElasticNet Regression": ElasticNet(max_iter=10000, random_state=RANDOM_STATE),
        "Decision Tree Regressor": DecisionTreeRegressor(random_state=RANDOM_STATE),
        "Random Forest Regressor": RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
        "Gradient Boosting Regressor": GradientBoostingRegressor(random_state=RANDOM_STATE),
        "Support Vector Regressor (SVR)": SVR(),
        "K-Nearest Neighbors Regressor": KNeighborsRegressor(),
    }[name]
    if params:
        base.set_params(**params)
    return base

ablation_rows = []
for name in candidate_models.keys():
    model_with = clone_with_best_params(name).fit(X_train_processed, y_train)
    model_without = clone_with_best_params(name).fit(X_train_without, y_train)

    pred_with = model_with.predict(X_test_full_processed)
    pred_without = model_without.predict(X_test_without)

    r2_with = r2_score(y_test_full, pred_with)
    r2_without = r2_score(y_test_without, pred_without)

    ablation_rows.append({
        "Model": name,
        "R2_with_gradient": r2_with,
        "R2_without_gradient": r2_without,
        "R2_Improvement": r2_with - r2_without
    })

# Polynomial regression, same degree in both arms
poly_with = Pipeline([
    ('poly', PolynomialFeatures(degree=best_degree, include_bias=False)),
    ('linear', LinearRegression())
]).fit(X_train_processed, y_train)
poly_without = Pipeline([
    ('poly', PolynomialFeatures(degree=best_degree, include_bias=False)),
    ('linear', LinearRegression())
]).fit(X_train_without, y_train)

ablation_rows.append({
    "Model": "Polynomial Regression",
    "R2_with_gradient": r2_score(y_test_full, poly_with.predict(X_test_full_processed)),
    "R2_without_gradient": r2_score(y_test_without, poly_without.predict(X_test_without)),
})
ablation_rows[-1]["R2_Improvement"] = (
    ablation_rows[-1]["R2_with_gradient"] - ablation_rows[-1]["R2_without_gradient"]
)

ablation_df = pd.DataFrame(ablation_rows).sort_values("R2_with_gradient", ascending=False)
ablation_df

In [ ]:
plt.figure(figsize=(14, 7))
plt.bar(ablation_df["Model"], ablation_df["R2_Improvement"])
plt.axhline(0, linewidth=1)
plt.xlabel("Regression Model")
plt.ylabel("Change in R² (same hyperparameters both arms)")
plt.title("Effect of Thermal_Gradient on R² (hyperparameter-matched, honest test set)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

 ## Summary of what changed and why it matters

 | Issue | Original behavior | Fix |
 |---|---|---|
 | Subject leakage | Round-based split; same subjects in train & test | `GroupShuffleSplit` on `Subject_ID` across all 4 rounds combined |
 | Model-selection leakage | "Best model" chosen from test-set R² ranking | Best model chosen via `GroupKFold` CV on train only; test touched once |
 | Fever filtering | Filtered `>39.5` from train *and* test, on the wrong column | Filtered only from train, using the actual target column; test kept full range, evaluated separately on fever subset |
 | Outlier bounds | IQR computed per round incl. test round | IQR computed on train only, applied to test |
 | Ablation study | Tuned model (with feature) vs. default model (without) | Same CV-selected hyperparameters used in both arms |

 Expect the reported R² here to be lower than the original notebook's — that drop is the
 size of the leakage that was previously inflating the numbers, not a regression in the
 modeling approach.